Ähnlich wie die Fusion der ICON Daten mit den Verkehrsdaten werden nun die MAIAC-Daten mit der aktuellen Masterdatei "Features_Verkehr_und_Wetter.csv" gemerged und wie auch zuvor die MAIAC-Daten basierend auf die Sensor-Koordinaten den korrekten Sensoren zugeordnet.

In [1]:
import os
import pandas as pd
import geopandas as gpd
import rasterio
import numpy as np

# =========================================================
# 1. Pfade definieren
# =========================================================
master_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Features_Verkehr_und_Wetter.csv"
sensor_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (Sensor_Community)\sensor_community_mit_kopfklinik_may_august.csv"
raster_base_dir = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (DOM)\MAIAC_AOD"

# Ausgabedatei
output_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Features_MAIAC_Verkehr_und_Wetter.csv"

# =========================================================
# 2. Master-Daten laden & Koordinaten sichern
# =========================================================
print("Lade Master-Datensatz...")
df_merged = pd.read_csv(master_file, sep=";", decimal=",")
df_merged["station_id"] = df_merged["station_id"].astype(str)

# SICHERHEITS-CHECK: Falls lat/lon im Master fehlen, holen wir sie aus der Sensordatei
if "lon" not in df_merged.columns or "lat" not in df_merged.columns:
    print("Koordinaten fehlen im Master. Hole sie aus der Sensor-Datei...")
    df_sensors = pd.read_csv(sensor_file, sep=";", decimal=",")
    for col in ["lat", "lon"]:
        df_sensors[col] = df_sensors[col].astype(str).str.replace(",", ".", regex=False)
        df_sensors[col] = pd.to_numeric(df_sensors[col], errors="coerce")

    unique_sensors = df_sensors[["sensor_id", "lat", "lon"]].drop_duplicates(subset=["sensor_id"]).dropna()
    unique_sensors["station_id"] = unique_sensors["sensor_id"].astype(float).astype(int).astype(str)
    unique_sensors = unique_sensors[["station_id", "lat", "lon"]]

    df_merged = pd.merge(df_merged, unique_sensors, on="station_id", how="left")
else:
    print("Koordinaten (lat/lon) sind bereits im Master vorhanden. Merge übersprungen!")

# Sicherstellen, dass Koordinaten numerisch sind
df_merged["lat"] = pd.to_numeric(df_merged["lat"], errors="coerce")
df_merged["lon"] = pd.to_numeric(df_merged["lon"], errors="coerce")

# =========================================================
# 3. Koordinatensystem für die Raster-Abfrage anpassen
# =========================================================
print("Transformiere Koordinaten in das metrische System (EPSG:25832)...")
gdf = gpd.GeoDataFrame(
    df_merged,
    geometry=gpd.points_from_xy(df_merged["lon"], df_merged["lat"]),
    crs="EPSG:4326"
)

# In das System der GeoTIFFs (UTM 32N) umwandeln
gdf = gdf.to_crs("EPSG:25832")
df_merged["x_utm"] = gdf.geometry.x
df_merged["y_utm"] = gdf.geometry.y

# =========================================================
# 5. MAIAC AOD aus den Rastern extrahieren
# =========================================================
print("Extrahiere MAIAC AOD Satellitendaten...")

df_merged["MAIAC_AOD"] = np.nan
unique_dates = df_merged["date"].unique()

for current_date in unique_dates:
    mask = df_merged["date"] == current_date
    day_points = df_merged.loc[mask, ["x_utm", "y_utm"]].values

    coords_list = [(x, y) for x, y in day_points]

    tif_filename = f"MAIAC_AOD_{current_date}.tif"
    tif_path = os.path.join(raster_base_dir, tif_filename)

    if os.path.exists(tif_path):
        with rasterio.open(tif_path) as src:
            nodata_val = src.nodata  # Liest den offiziellen NoData-Wert aus der Datei

            sampled_values = []
            for val in src.sample(coords_list):
                v = val[0]

                # FILTER: Wenn der Wert NoData ist, absurd groß (> 10) oder negativ (< 0),
                # dann ist es keine gültige AOD-Messung (Wolken etc.)
                if v == nodata_val or v < 0 or v > 10:
                    sampled_values.append(np.nan)
                else:
                    sampled_values.append(v)

            df_merged.loc[mask, "MAIAC_AOD"] = sampled_values
    else:
        print(f"Warnung: Keine MAIAC-Daten für {current_date}")

# =========================================================
# 5. Aufräumen und Speichern
# =========================================================
# Metrische Hilfskoordinaten fallen lassen, um die Datei sauber zu halten
#df_final = df_merged.drop(columns=["x_utm", "y_utm"], errors="ignore")

df_merged.to_csv(output_file, index=False, sep=";", decimal=",")

print("-" * 60)
print(f"ERFOLG! Der Datensatz inklusive MAIAC AOD liegt hier:\n{output_file}")
print("-" * 60)

Lade Master-Datensatz...
Koordinaten (lat/lon) sind bereits im Master vorhanden. Merge übersprungen!
Transformiere Koordinaten in das metrische System (EPSG:25832)...
Extrahiere MAIAC AOD Satellitendaten...
------------------------------------------------------------
ERFOLG! Der Datensatz inklusive MAIAC AOD liegt hier:
C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Features_MAIAC_Verkehr_und_Wetter.csv
------------------------------------------------------------
